In [1]:
import requests
import pandas as pd

metadata_url = "https://data.cityofnewyork.us/api/views/erm2-nwe9.json"

response = requests.get(metadata_url, timeout=30)
response.raise_for_status()

metadata = response.json()

columns_info = []

for column in metadata["columns"]:
    field_name = column.get("fieldName")
    
    if field_name and not field_name.startswith(":"):
        columns_info.append({
            "Column Name": column.get("name"),
            "API Field Name": field_name,
            "Description": column.get("description", "")
        })

schema_df = pd.DataFrame(columns_info)

print("Number of available attributes:", len(schema_df))
display(schema_df)


Number of available attributes: 44


,Column Name,API Field Name,Description
0,Unique Key,unique_key,Unique identifier of a Service Request (SR) in...
1,Created Date,created_date,Date SR was created\n
2,Closed Date,closed_date,Date SR was closed by responding agency\n
3,Agency,agency,Acronym of responding City Government Agency\n
4,Agency Name,agency_name,Full Agency name of responding City Government...
5,Problem (formerly Complaint Type),complaint_type,This is the first level of a hierarchy identif...
6,Problem Detail (formerly Descriptor),descriptor,This is associated to the Problem (formerly Co...
7,Additional Details,descriptor_2,A third level of detail about the Problem (for...
8,Location Type,location_type,Describes the type of location used in the add...
9,Incident Zip,incident_zip,"Incident location zip code, provided by geo va..."


In [2]:
import pandas as pd

df = pd.read_csv("nyc_311_stratified_sample_v2.csv")

print("Rows:", len(df))
print("Unique IDs:", df["unique_key"].nunique())
print("Current columns:", len(df.columns))
print("Missing IDs:", df["unique_key"].isna().sum())

display(df[["unique_key", "created_date", "borough"]].head())


Rows: 60000
Unique IDs: 60000
Current columns: 10
Missing IDs: 0


,unique_key,created_date,borough
0,58747927,2023-09-07 21:18:18,BROOKLYN
1,63535037,2024-12-27 15:50:00,BROOKLYN
2,63889795,2025-01-25 20:31:17,BROOKLYN
3,57928606,2023-06-18 01:19:59,QUEENS
4,57899699,2023-06-14 18:14:07,MANHATTAN


In [3]:
import pandas as pd
import requests
import time
import os
import json

URL = "https://data.cityofnewyork.us/resource/erm2-nwe9.json"

# Load the original sample
df = pd.read_csv("nyc_311_stratified_sample_v2.csv")
df["unique_key"] = df["unique_key"].astype(str)

# All 44 fields from the dataset schema
all_columns = [
    "unique_key", "created_date", "closed_date", "agency",
    "agency_name", "complaint_type", "descriptor", "descriptor_2",
    "location_type", "incident_zip", "incident_address", "street_name",
    "cross_street_1", "cross_street_2", "intersection_street_1",
    "intersection_street_2", "address_type", "city", "landmark",
    "facility_type", "status", "due_date", "resolution_description",
    "resolution_action_updated_date", "community_board",
    "council_district", "police_precinct", "bbl", "borough",
    "x_coordinate_state_plane", "y_coordinate_state_plane",
    "open_data_channel_type", "park_facility_name", "park_borough",
    "vehicle_type", "taxi_company_borough", "taxi_pick_up_location",
    "bridge_highway_name", "bridge_highway_direction", "road_ramp",
    "bridge_highway_segment", "latitude", "longitude", "location"
]

checkpoint_dir = "nyc311_full_attributes_checkpoints_100"
os.makedirs(checkpoint_dir, exist_ok=True)

batch_size = 100
session = requests.Session()
failed_batches = []
total_batches = (len(df) + batch_size - 1) // batch_size

for start in range(0, len(df), batch_size):
    batch_number = start // batch_size
    checkpoint_path = os.path.join(
        checkpoint_dir, f"batch_{batch_number:04d}.json"
    )

    if os.path.exists(checkpoint_path):
        continue

    batch_ids = df["unique_key"].iloc[start:start + batch_size].tolist()
    id_list = ",".join(f"'{key}'" for key in batch_ids)

    success = False

    for attempt in range(3):
        try:
            response = session.get(
                URL,
                params={
                    "$where": f"unique_key IN ({id_list})",
                    "$select": ",".join(all_columns),
                    "$limit": batch_size
                },
                timeout=60
            )

            if response.status_code == 200:
                records = response.json()

                returned_ids = {
                    str(record["unique_key"]) for record in records
                }

                if returned_ids == set(batch_ids):
                    with open(checkpoint_path, "w", encoding="utf-8") as f:
                        json.dump(records, f, ensure_ascii=False)

                    success = True
                    break

                print(
                    f"Batch {batch_number}: ID mismatch "
                    f"(expected {len(batch_ids)}, got {len(records)})"
                )
                break

            print(
                f"Batch {batch_number}, attempt {attempt + 1}: "
                f"HTTP {response.status_code}"
            )
            time.sleep(2)

        except requests.RequestException as error:
            print(f"Batch {batch_number}, attempt {attempt + 1}: {error}")
            time.sleep(2)

    if not success:
        failed_batches.append(batch_number)

    completed = min(start + batch_size, len(df))
    if (batch_number + 1) % 20 == 0 or completed == len(df):
        print(
            f"Progress: {completed}/{len(df)} IDs processed; "
            f"failed batches: {len(failed_batches)}"
        )

print("Extraction pass finished.")
print("Failed batches:", failed_batches)
print("Checkpoint folder:", checkpoint_dir)


Extraction pass finished.
Failed batches: []
Checkpoint folder: nyc311_full_attributes_checkpoints_100


In [4]:
import pandas as pd
import json
import os
import glob

# Load the original sample IDs
original = pd.read_csv(
    "nyc_311_stratified_sample_v2.csv",
    dtype={"unique_key": str}
)
expected_ids = original["unique_key"].tolist()

# Read all saved checkpoint batches
checkpoint_dir = "nyc311_full_attributes_checkpoints_100"
files = sorted(glob.glob(os.path.join(checkpoint_dir, "batch_*.json")))

records = []

for file in files:
    with open(file, "r", encoding="utf-8") as f:
        records.extend(json.load(f))

print("Checkpoint files:", len(files))
print("Total retrieved records:", len(records))

# Build a DataFrame with all 44 schema columns
full_df = pd.DataFrame(records)

full_df["unique_key"] = full_df["unique_key"].astype(str)

all_columns = [
    "unique_key", "created_date", "closed_date", "agency",
    "agency_name", "complaint_type", "descriptor", "descriptor_2",
    "location_type", "incident_zip", "incident_address", "street_name",
    "cross_street_1", "cross_street_2", "intersection_street_1",
    "intersection_street_2", "address_type", "city", "landmark",
    "facility_type", "status", "due_date", "resolution_description",
    "resolution_action_updated_date", "community_board",
    "council_district", "police_precinct", "bbl", "borough",
    "x_coordinate_state_plane", "y_coordinate_state_plane",
    "open_data_channel_type", "park_facility_name", "park_borough",
    "vehicle_type", "taxi_company_borough", "taxi_pick_up_location",
    "bridge_highway_name", "bridge_highway_direction", "road_ramp",
    "bridge_highway_segment", "latitude", "longitude", "location"
]

# Add schema columns that were absent from API responses
# because their values were null in every returned record.
full_df = full_df.reindex(columns=all_columns)

# Check IDs
duplicate_count = full_df["unique_key"].duplicated().sum()
retrieved_ids = set(full_df["unique_key"])
expected_id_set = set(expected_ids)

missing_ids = expected_id_set - retrieved_ids
unexpected_ids = retrieved_ids - expected_id_set

print("Rows:", len(full_df))
print("Unique IDs:", full_df["unique_key"].nunique())
print("Duplicate IDs:", duplicate_count)
print("Missing expected IDs:", len(missing_ids))
print("Unexpected IDs:", len(unexpected_ids))
print("Columns:", len(full_df.columns))

# Save only when the sample matches exactly
if (
    len(full_df) == len(expected_ids)
    and duplicate_count == 0
    and not missing_ids
    and not unexpected_ids
):
    # Restore original sample order
    full_df = full_df.set_index("unique_key").loc[expected_ids]
    full_df = full_df.reset_index()

    output_file = "nyc_311_full_attributes_v1.csv"
    full_df.to_csv(output_file, index=False)

    print("\nVALIDATION PASSED")
    print("Saved:", output_file)
    print("Final shape:", full_df.shape)
else:
    print("\nVALIDATION FAILED — no final CSV was saved.")
    print("Please share the output so we can investigate.")


Checkpoint files: 600
Total retrieved records: 60000
Rows: 60000
Unique IDs: 60000
Duplicate IDs: 0
Missing expected IDs: 0
Unexpected IDs: 0
Columns: 44

VALIDATION PASSED
Saved: nyc_311_full_attributes_v1.csv
Final shape: (60000, 44)
